# 03 — Black–Litterman

`BlackLitterman` (in `maplab.models`): a long-only tangency portfolio on a Black–Litterman posterior mean (equal-weight prior, 12−1 momentum sign views), scored by the rolling backtest against EW and 60/40. It is return-based: it needs μ, here built from a prior and views instead of a sample mean.

## §1 Black–Litterman and the idea

Black–Litterman replaces an estimated μ with a prior implied by a reference portfolio, the returns that make it mean-variance optimal, and blends in explicit views with stated confidence. Notebook 01 showed that a sample-mean μ makes the optimizer an error maximizer, and notebook 02 that a CAPM μ collapses it to 100% SPY. Here the prior is implied by EW, so with no views the model holds EW. The views are the signs of trailing 12-month excess returns (time-series momentum) with the latest month skipped, and each view is as uncertain as the prior on that asset (the He–Litterman choice).

## §2 The model: prior, views, posterior

**General form.** Returns have an uncertain mean, $\mu \sim N(\Pi, \tau\Sigma)$, and views state $Q = P\mu + \varepsilon$ with $\varepsilon \sim N(0, \Omega)$. The posterior mean is

$$\mu_{BL} = \left[(\tau\Sigma)^{-1} + P'\Omega^{-1}P\right]^{-1}\left[(\tau\Sigma)^{-1}\Pi + P'\Omega^{-1}Q\right] = \Pi + \tau\Sigma P'(P\tau\Sigma P' + \Omega)^{-1}(Q - P\Pi),$$

with posterior covariance of the mean $M = \left[(\tau\Sigma)^{-1} + P'\Omega^{-1}P\right]^{-1}$. Precise views (small $\Omega$) pull $\mu_{BL}$ toward $Q$; vague views leave it at $\Pi$.

**Prior.** $\Pi = \delta\,\Sigma\,w_{EW}$ with $\delta = \mathrm{sr}_{ref}/\sigma_{EW}$, so EW's implied Sharpe ratio is $\mathrm{sr}_{ref} = 0.3$, fixed a priori. The long-only tangency on $\Pi$ is $w_{EW}$ exactly, whatever $\Sigma$ is.

**Views.** One absolute view per asset ($P = I$): $Q = \Pi + k\, s \circ \sigma$, where $s_i$ is the sign of asset $i$'s trailing 12−1 excess return and $\sigma_i$ its volatility. Each view moves an asset's Sharpe ratio $k$ above or below the prior's.

**Confidence.** $\Omega = \tau D$ with $D = \mathrm{diag}(\Sigma)$ (He–Litterman): each view is as uncertain as the prior on that asset. Then

$$\mu_{BL} = \Pi + k\,\Sigma(\Sigma + D)^{-1}(s \circ \sigma), \qquad \Sigma^{-1}\mu_{BL} = \delta\, w_{EW} + k\,(\Sigma + D)^{-1}(s \circ \sigma).$$

$\tau$ cancels, and $k = 0$ returns EW. The unconstrained tangency is EW plus a momentum tilt, sized relative to EW by $k\,\sigma_{EW}/\mathrm{sr}_{ref}$. If $\Sigma$ were diagonal, $\mu_{BL}$ would sit halfway between $\Pi$ and $Q$.

**The optimization problem.** The unconstrained tangency shorts and, as k grows, levers up (§4.4), so the class solves the long-only version on the same 252-day $\Sigma$:

$$\max_w \; \frac{w'\mu_{BL}}{\sqrt{w'\Sigma w}} \quad \text{s.t.} \quad \mathbf{1}'w = 1, \; w \ge 0 .$$

$\mu_{BL}$ is already an excess return, so no $r_f$ enters. Weights use $\Sigma$, not $\Sigma + M$ with $M = \tau\Sigma(\Sigma + D)^{-1}D$; $M$ scales with $\tau$, so using $\Sigma$ keeps $\tau$ out of the weights.

**k.** $k = 0.2$ was planned. A snapshot check at three train-window dates showed $k \ge 0.2$ pushes effective N below 5 at 2022-12-31, so $k = 0.1$ was fixed before the backtest and $0.2$ kept as a sensitivity (§4.4).

**Inputs.** $\Sigma$ and momentum come from the same trailing 252 daily log returns before the rebalance date; momentum sums the first 231 of them (days $t-252$ to $t-22$) in excess of BIL via $\log(1 + r_f)$.

## Setup

Prices, log returns for estimation, simple returns for the backtest, BIL as $r_f$, and the panel the classes read.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import maplab as ml
from maplab import Panel, apply_style, BlackLitterman, EqualWeight, FixedWeight, backtest, summary, tangency_closed_form

from helpers import nb03 as h

In [ ]:
apply_style()
prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # estimation (Sigma, momentum)
simple_returns = ml.to_simple_returns(prices)  # compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

## §3 The class and the rolling backtest

`BlackLitterman` takes Σ from the shared `Strategy._estimate` (defined in notebook 01 §3); `posterior` in `src/maplab/models.py` builds Π, the views and μ_BL, and `predict_weights` solves the §2 problem on μ_BL.

**Solver.** The long-only tangency shared with MaxSharpe (notebook 01 §3). If no asset has $\mu_{BL} > 0$ the class falls back to GMV and logs the date; rebalances with effective N below 5 are logged as concentrated, with the weights unchanged.

**Backtest** as in notebook 01 §3.

## §4 Inputs and snapshot

One rebalance, 2022-12-31: the prior, the views, the posterior and the weights.

### §4.1 Why not a CAPM prior

A prior implied by 100% SPY, $\Pi = \delta\,\Sigma\,e_{SPY}$, gives $\Sigma^{-1}\Pi \propto e_{SPY}$: the no-view portfolio is 100% SPY, the collapse of notebook 02 §4.2.

In [ ]:
rebal_le_split = ml.rebalance_dates(log_returns.index)
asof_split = rebal_le_split[rebal_le_split <= split_ts].max()

bl_probe = BlackLitterman()
post_probe = bl_probe.posterior(panel, asof_split)
Sigma_probe = post_probe["Sigma"]
delta_probe = post_probe["delta"]

e_spy = pd.Series(0.0, index=ml.UNIVERSE)
e_spy["SPY"] = 1.0
Pi_capm = delta_probe * (Sigma_probe.to_numpy() @ e_spy.to_numpy())
Pi_capm = pd.Series(Pi_capm, index=ml.UNIVERSE)

w_capm_tangency = tangency_closed_form(Pi_capm, Sigma_probe, 0.0)
print(f"CAPM-implied prior -> unconstrained tangency SPY weight: {w_capm_tangency['SPY']:.6f}")
print(f"max |other assets|: {w_capm_tangency.drop('SPY').abs().max():.2e}")

**Reading.** The CAPM prior's tangency is 100% SPY, every other weight at most 2.04e-15. With that prior the views would be read against SPY instead of the EW benchmark.

### §4.2 Posterior at the snapshot

Π, σ, 12−1 excess momentum, its sign, Q and μ_BL at k = 0.1 (annualized, %). Then the §2 checks: k = 0 returns EW, τ cancels, the closed form holds. Last, the unconstrained tangency on μ_BL: how hard the views pull before the long-only constraint.

In [ ]:
bl01 = BlackLitterman(k=0.1)
post = bl01.posterior(panel, asof_split)

mech_table = h.mechanics_table(post).rename(columns={
    "Pi_%": "Π (%)", "sigma_%": "σ (%)", "mom12-1_%": "momentum 12−1 (%)", "Q_%": "Q (%)", "mu_BL_%": "μ_BL (%)"})
print(f"δ = {post['delta']:.2f}, negative Π: {int((post['pi'] < 0).sum())}")
ml.reporting.show_table(mech_table, dp=2, sign_cols=["sign"])

In [ ]:
chk = h.design_checks(panel, asof_split, post)
w_unc = chk["w_unc"]
longs = w_unc.sort_values(ascending=False).head(3)
shorts = w_unc[w_unc < 0].sort_values().head(3)


def signed(s):
    return ", ".join(f"{t} {v:.1%}".replace("-", "−") for t, v in s.items())


print(f"checks: k = 0 gives EW to {chk['k0_dev']:.1e}; τ invariance {chk['tau_dev']:.1e}; "
      f"closed-form identity {chk['identity_dev']:.1e}")
print(f"unconstrained BL(0.1) tangency: gross {w_unc.abs().sum():.2f}×; longs {signed(longs)}; "
      f"shorts {signed(shorts) if len(shorts) else 'none'}")

**Reading.** Only UUP has a negative prior (Π = −1.8%): Π is proportional to each asset's covariance with EW, and UUP moves against the EW book. Eleven of 13 momentum signs are negative; only DBC and UUP are positive. The §2 algebra holds numerically, and the unconstrained tangency is a mild tilt at k = 0.1 (1.13× gross).

### §4.3 Prior vs posterior

Π and μ_BL for k = 0.1 and 0.2 at 2022-12-31.

In [ ]:
fig = h.prior_posterior_figure(panel, asof_split)
plt.show()

**Reading.** The views pull the eleven negative-momentum assets down and DBC and UUP up. UUP's posterior stays negative (−1.03%), yet BL(0.1) still holds it (§4.4): a held asset with a negative expected return earns its place by hedging the rest of the portfolio. With no cash asset, a long-only book can express this risk-off tilt only through DBC and UUP.

### §4.4 k as a concentration knob

Effective N, largest weight and unconstrained gross leverage for k = 0.1, 0.2 and 0.4, then the long-only weights against EW. k = 0.4 is snapshot only, never backtested.

In [ ]:
k_table, wtab = h.k_sensitivity(panel, asof_split)

k_shown = k_table.rename(columns={"effN": "effective N", "max_w": "largest weight (%)", "argmax": "largest holding",
                                  "unconstrained_gross_leverage": "gross leverage"})
k_shown = ml.reporting.show_table(k_shown, pct=["largest weight (%)"], dp=2)
k_shown["gross leverage"] = k_shown["gross leverage"].map(lambda v: f"{v:.2f}×")
display(k_shown)

In [ ]:
fig = h.ew_vs_bl_weights(wtab)
plt.show()

**Reading.** Effective N falls 7.58 → 3.47 → 1.78 and unconstrained gross leverage rises 1.13× → 2.04× → 68.9× as k goes 0.1 → 0.2 → 0.4. Eleven negative signs are one risk-off bet, not 13 signals, and a fully invested long-only book can place it only in DBC and UUP.

### §4.5 Capital vs risk

Capital weights (dark) and shares of portfolio variance (light) at 2022-12-31; a negative share is a hedge. Table columns as in notebook 01 §4.5.

In [ ]:
SNAP = asof_split
Sigma_2022 = post["Sigma"]
w_2022 = {
    "BL(0.1)": wtab["k=0.1"],
    "BL(0.2)": wtab["k=0.2"],
    "EqualWeight": EqualWeight()(panel, SNAP),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40")(panel, SNAP),
}
w_2022 = {name: w.reindex(Sigma_2022.columns) for name, w in w_2022.items()}

weights_c5 = {"BL(0.1)": w_2022["BL(0.1)"], "BL(0.2)": w_2022["BL(0.2)"],
              "EW": w_2022["EqualWeight"], "60/40": w_2022["60/40"]}
div = ml.diagnostics.diversification_table(weights_c5, Sigma_2022).rename(columns={
    "w_UUP": "UUP weight (%)", "effN": "effective N", "n_uncorr": "uncorrelated bets",
    "max_RC_share": "largest risk share (%)", "DR": "diversification ratio", "exante_vol": "ex-ante vol (%)"})
display(ml.reporting.show_table(div, pct=["UUP weight (%)", "largest risk share (%)", "ex-ante vol (%)"], dp=2))

fig = ml.plotting.capital_vs_risk(weights_c5, Sigma_2022, title="Capital weight vs risk share, 2022-12-31 (sample Σ)")
plt.show()

**Reading.** BL(0.1) holds 19.8% UUP and spreads over 2.18 uncorrelated bets, BL(0.2) 41.8% and 3.28 (EW 1.40, 60/40 1.76). The largest single variance share is 38% for BL(0.1) and 84% for BL(0.2).

## §5 Out-of-sample results

Walk-forward as in notebook 01 §5. BL(0.1) is the model and BL(0.2) its sensitivity; EW is BL with k = 0. k was chosen from a snapshot check (§2), so the test window is the cleaner read. The last cell counts fallbacks, retries and concentrated rebalances (effN < 5); `tests/test_nb01_03_checks.py` rebuilds these runs and checks their Sharpe ratios against `registrations/reproduction.toml`.

In [ ]:
strategies = {
    "BL(0.1)": BlackLitterman(k=0.1),
    "BL(0.2)": BlackLitterman(k=0.2),
    "EqualWeight": EqualWeight(),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40"),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

In [ ]:
rows = []
for name, r in results.items():
    net = r["net"]
    turnover = r["diag"]["turnover_per_rebalance"]
    win_defs = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        s = summary(net.loc[net_sel], turnover.loc[t_sel], rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]

full = summary_table.xs("full", level="window")
train = summary_table.xs("train (<= split)", level="window")
test = summary_table.xs("test (> split)", level="window")
overview = pd.DataFrame({
    "Sharpe full": full["sharpe"], "Sharpe train": train["sharpe"], "Sharpe test": test["sharpe"],
    "return (%)": full["ann_return"], "vol (%)": full["ann_vol"],
    "max DD (%)": full["max_dd"], "turnover (%/yr)": full["ann_turnover"],
}).rename(index={"EqualWeight": "EW"}).rename_axis(None)
ml.reporting.show_table(overview, pct=["return (%)", "vol (%)", "max DD (%)", "turnover (%/yr)"],
                        pct_dp=1, dp=2, int_cols=["turnover (%/yr)"])

In [ ]:
counts = ", ".join(
    f"{n} {len(strategies[n].fallback_dates)}/{len(strategies[n].retry_dates)}/{len(strategies[n].concentration_dates)}"
    for n in ("BL(0.1)", "BL(0.2)"))
print(f"fallback/retry/concentrated (effN < 5): {counts} of {results['BL(0.1)']['diag']['n_rebalances']}")
print("BL(0.1) concentrated dates:", ", ".join(str(d.date()) for d in strategies["BL(0.1)"].concentration_dates))

**Reading.** Full-sample Sharpe: 60/40 0.91, EW 0.74, BL(0.1) 0.64, BL(0.2) 0.58, and the order holds from 2023 (1.00, 0.91, 0.70, 0.50). BL(0.1) runs at 6.9% volatility against EW's 7.9% and turns over 239% a year against 14%. Effective N fell below 5 at 16 of 208 rebalances for k = 0.1 (mostly in clusters around 2013–14, 2018 and 2023) and 66 for k = 0.2, with no fallback or retry.

## §6 What drove the results

### §6.1 Weights by asset group

Target weights by group at every rebalance (dotted line: split). EW and 60/40 are fixed and appear only in the table.

In [ ]:
fig = ml.plotting.group_stackplot(results, ["BL(0.1)", "BL(0.2)"], figsize=(9, 8), split_ts=split_ts)
plt.show()

group_means = pd.DataFrame({
    name: ml.diagnostics.weights_by_group(results[name]["wlog"]).mean()
    for name in ["BL(0.1)", "BL(0.2)", "EqualWeight", "60/40"]
}).rename_axis(None).rename(columns={"EqualWeight": "EW"}).rename(index=ml.plotting.GROUP_LABELS)
ml.reporting.show_table(group_means, pct=list(group_means.columns))

**Reading.** On average BL(0.1) holds 22% equities and 10% fx, against 23% and 8% for EW. The views add most to credit (+4 pp) and take most from real assets (−4 pp).

### §6.2 Concentration and turnover

Effective N, largest weight, UUP weight and trailing-12-rebalance turnover at every rebalance. BL(0.2) is the dashed orange line; EW is the grey dashed line at 13 in the top panel and at the bottom of the turnover panel; the grey dashed line at 5 is the effective-N floor; 60/40 is the solid teal line.

In [ ]:
c3_names = ["BL(0.1)", "BL(0.2)", "60/40"]
colors = {name: ml.plotting.run_color(name) for name in c3_names}
fig = ml.plotting.concentration_panel(
    results, c3_names,
    rows=("effN", "max_w", "highlight", "turnover"), ref="EqualWeight", ref_label="EW", split_ts=split_ts,
    figsize=(9.5, 14), colors=colors, effn_floor=5.0,
    suptitle="BL(0.1) and BL(0.2) vs 60/40: effective N, largest weight, UUP weight, turnover",
)
plt.show()

med_effn = {n: float(results[n]["wlog"].apply(ml.diagnostics.effective_n, axis=1).median())
            for n in ["BL(0.1)", "BL(0.2)"]}
print("median effective N: " + "  ".join(f"{n}={v:.2f}" for n, v in med_effn.items()))

**Reading.** Median effective N is 7.60 for BL(0.1) and 6.44 for BL(0.2). Turnover is 239% a year for BL(0.1) and 270% for BL(0.2), against 14% for EW and 11% for 60/40.

### §6.3 What the views did: BL − EW

With k = 0 the model is EW, so BL − EW isolates the views. First each strategy's ex-post CAPM regression (daily simple excess return on SPY − BIL, plain OLS), then the split of the active return into a β part and an α part. Exploratory; notebook 08 tests these gaps.

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_strategy_table = ml.inference.capm_table(results, ["BL(0.1)", "BL(0.2)", "EqualWeight", "60/40"], MKT_simple, rf_daily)
capm_shown = capm_strategy_table.rename(
    columns={"alpha_ann": "α (%/yr)", "t_alpha": "t(α)", "beta": "β", "r2": "R²"},
    index={"EqualWeight": "EW"}).rename_axis(None)
ml.reporting.show_table(capm_shown, pct=["α (%/yr)"], pct_dp=2, dp=2, int_cols=["n"])

**Reading.** Realized β is 0.29 for BL(0.1), 0.23 for BL(0.2), 0.36 for EW and 0.55 for 60/40. No α is large: t = 0.14 for BL(0.1), 0.32 for BL(0.2), 0.49 for EW and 1.66 for 60/40.

In [ ]:
bl_pairs = [("BL(0.1)", "EqualWeight"), ("BL(0.2)", "EqualWeight")]
bl_paired, bl_paired_series = ml.inference.paired_table(results, bl_pairs, split_ts)
bl_diff_capm = ml.inference.diff_regression(bl_paired_series, bl_pairs, MKT_simple)
decomp = ml.inference.beta_decomposition(capm_strategy_table, bl_paired, bl_diff_capm, bl_pairs,
                                         results, rf_daily, MKT_simple)

decomp_rows = {f"{name} − EW": decomp[f"{name} - {ref}"] for name, ref in bl_pairs}
decomp_shown = pd.DataFrame(decomp_rows).T[["d_alpha", "d_beta", "beta_part", "realized_diff_full"]].astype(float).rename(
    columns={"d_alpha": "Δα (%/yr)", "d_beta": "Δβ", "beta_part": "β part (%/yr)", "realized_diff_full": "active return (%/yr)"})
ml.reporting.show_table(decomp_shown, pct=["Δα (%/yr)", "β part (%/yr)", "active return (%/yr)"], pct_dp=2, dp=4)

**Reading.** BL(0.1) trails EW by 1.36% a year over the full sample: lower β explains −0.96% (Δβ −0.0655 times the market's excess return) and α the other −0.40%, which includes about 0.23% of extra trading cost (239% against 14% turnover at 10 bp). At k = 0.2 the gap is 1.97%, of which the β part is −1.80%.

Cumulative BL − EW (solid: β-adjusted; dashed: total). Exploratory.

In [ ]:
ew_pairs = [(a, "EW") for a, _ in bl_pairs]          # display name "EW" for the EqualWeight run
ew_series = {(a, "EW"): bl_paired_series[(a, b)] for a, b in bl_pairs}
fig = ml.plotting.cumulative_paired(
    ew_series, ew_pairs, split_ts,
    beta={f"{a} - EW": bl_diff_capm.loc[f"{a} - {b}", "beta_diff"] for a, b in bl_pairs},
    mkt_excess=MKT_simple,
    colors={"BL(0.1) - EW": ml.plotting.RUN_COLORS["BL"],
            "BL(0.2) - EW": "#a84f00"},
    title="Exploratory: cumulative BL − EW, solid = β-adjusted, dashed = total",
)
plt.show()

**Reading.** The dashed lines end near 17.2 years times the active returns above (about −23% and −34%), the β-adjusted lines near 17.2 years times Δα (about −7% and −3%): most of the gap is β. BL(0.1)'s β-adjusted gap was positive from about 2015 to mid-2019 and has been negative since 2020.

## §7 TAKEAWAY — momentum views on an EW prior lower β and add turnover, not α

- **No views, no bets.** The EW-implied prior makes k = 0 hold EW; τ cancels, and k sets the size of the momentum tilt (§2).
- **k is a concentration knob.** At 2022-12-31, 11 of 13 signs were negative: one risk-off bet that a fully invested long-only book can hold only in DBC and UUP; effective N falls 7.58 → 3.47 → 1.78 for k = 0.1, 0.2, 0.4.
- **Against the benchmarks:** full-sample Sharpe 0.64 against 0.74 for EW and 0.91 for 60/40 (0.70 against 0.91 and 1.00 from 2023), with 239% turnover a year against 14% and 11%.
- **The views mostly cut β.** BL(0.1) trails EW by 1.36% a year, −0.96% of it from lower β and about 0.23% from trading cost; at k = 0.2 the β part is −1.80% of −1.97%.
- **Concentration is episodic.** Effective N fell below 5 at 16 of 208 rebalances (66 at k = 0.2), with no fallbacks.

The risk-off view needs a cash leg this long-only, fully invested book lacks. Notebook 04 keeps the models and changes Σ.